In [1]:
!pip install -q mne matplotlib
import mne; print(mne.__version__)

1.12.1


In [2]:
from google.colab import files
uploaded = files.upload()

Saving eeg_prep_hedres to eeg_prep_hedres (4)


In [ ]:
%matplotlib inline

import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from google.colab import files

# -------------------------
# Load data
# -------------------------
path = "/content/eeg_prep_hedres"
data = np.loadtxt(path, skiprows=1)
eeg_data = data[:, 1:].T

names = [
    "FPz", "EOG1", "F3", "Fz", "F4", "EOG2", "FC5", "FC1", "FC2", "FC6",
    "T7", "C3", "C4", "Cz", "T8", "CP5", "CP1", "CP2", "CP6", "P7",
    "P3", "Pz", "P4", "P8", "PO7", "PO3", "POz", "PO4", "PO8", "O1", "Oz", "O2"
]

sfreq = 128.0
raw = mne.io.RawArray(
    eeg_data * 1e-6,
    mne.create_info(names, sfreq, ["eog" if "EOG" in n else "eeg" for n in names])
)
raw.rename_channels({"FPz": "Fpz"})
raw.set_montage(mne.channels.make_standard_montage("standard_1020"), on_missing="warn")

eeg_only = mne.pick_types(raw.info, eeg=True)
info_for_map = mne.pick_info(raw.info, eeg_only)
eeg_names = [raw.ch_names[i] for i in eeg_only]

# -------------------------
# Figure settings
# -------------------------
plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "savefig.dpi": 300,
})

t_start, t_end = 0, 10          # trace window (seconds)
topo_times = [1.0, 4.1, 8.0]    # topomap times (seconds)
offset_uv = 80                  # vertical spacing between traces (µV)

# -------------------------
# Build combined figure
# -------------------------
fig = plt.figure(figsize=(12, 8))
gs = gridspec.GridSpec(
    2, len(topo_times),
    height_ratios=[2.2, 1.2],
    hspace=0.35,
    wspace=0.25
)

# --- Top panel: EEG traces (span full width) ---
ax_trace = fig.add_subplot(gs[0, :])

t = raw.times
mask = (t >= t_start) & (t <= t_end)
t_plot = t[mask]

for i, ch_idx in enumerate(eeg_only):
    trace = raw.get_data(picks=[ch_idx])[0, mask] * 1e6
    ax_trace.plot(t_plot, trace + i * offset_uv, color="k", linewidth=0.5)
    ax_trace.text(t_start - 0.35, i * offset_uv, eeg_names[i],
                  ha="right", va="center", fontsize=7)

# Mark topomap times
for tt in topo_times:
    ax_trace.axvline(tt, color="crimson", linestyle="--", linewidth=1, alpha=0.8)

ax_trace.set(xlim=(t_start, t_end), yticks=[], xlabel="Time (s)")
ax_trace.set_title("Preprocessed EEG")

# --- Bottom panels: topomaps ---
topo_vals = []
for tt in topo_times:
    s = int(tt * sfreq)
    topo_vals.append(raw.get_data(picks=eeg_only)[:, s] * 1e6)
topo_vals = np.array(topo_vals)
vmin, vmax = np.percentile(topo_vals, [5, 95])

im = None
for i, (tt, vals) in enumerate(zip(topo_times, topo_vals)):
    ax_topo = fig.add_subplot(gs[1, i])
    im, _ = mne.viz.plot_topomap(
    vals,
    info_for_map,
    axes=ax_topo,
    show=False,
    vlim=(vmin, vmax),   # single tuple, no separate vmin= / vmax= args
    contours=6,
    sensors=True,
)
    ax_topo.set_title(f"{tt:.1f} s")

# Shared colorbar for topomaps
cbar_ax = fig.add_axes([0.92, 0.12, 0.015, 0.22])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Amplitude (µV)")

# Save + show + download
fig.savefig("eeg_trace_topomap_figure.pdf", bbox_inches="tight")
fig.savefig("eeg_trace_topomap_figure.png", dpi=300, bbox_inches="tight")
plt.show()

files.download("eeg_trace_topomap_figure.pdf")